In [ ]:
# Nesting Exception Handlers
    # Python allows try statements to be nested in two ways:
    # 1. Syntactically: one try statement is physically inside another.
    # 2. At runtime: one try calls a function that contains another try.
    # Python keeps track of active try statements using a LIFO stack (Last In, First Out).
    # When an exception is raised, Python returns to the most recently entered try statement that has a matching except clause.
# 1. Nested try/except combinations
try:
    func1()
except E:
    ...
def func1():
    try:
        func2()
    except E:
        ...
def func2():
    ...
    raise E
    ...
    # How it works:
    # 1. The outer try calls func1().
    # 2. func1() enters its own try and calls func2().
    # 3. func2() raises exception E.
    # 4. Python returns to the most recently entered matching try — the one inside func1().
    # 5. Its except E handles the exception.
    # 6. Execution continues after that try inside func1().
    # The outer except E does not execute because the exception has already been caught.
# example:
def func2():
    print('Inside func2')
    raise ValueError('Something went wrong')

def func1():
    try:
        print('Inside func1: calling func2')
        func2()
    except ValueError:
        print('Exception caught in func1')

    print('Continuing inside func1')

try:
    print('Inside outer try: calling func1')
    func1()
except ValueError:
    print('Exception caught in outer try')

print('Program continues')
    # Inside outer try: calling func1
    # Inside func1: calling func2
    # Inside func2
    # Exception caught in func1
    # Continuing inside func1
    # Program continues
# 2. Nested try/finally combinations
try:
    func1()
finally:
    ...

def func1():
    try:
        func2()
    finally:
        ...

def func2():
    ...
    raise E
    ...
    # Here, there are no except handlers. Each finally block executes as the exception propagates outward.
    # How it works:
    # 1. The outer try calls func1().
    # 2. func1() enters its try and calls func2().
    # 3. func2() raises exception E.
    # 4. Python executes the finally inside func1().
    # 5. The exception continues propagating outward.
    # 6. Python executes the outer finally.
    # 7. The exception eventually reaches the top-level handler if nobody catches it.
# example:
def func2():
    print('Inside func2')
    raise ValueError('Something went wrong')

def func1():
    try:
        print('Inside func1: calling func2')
        func2()
    finally:
        print('Finally in func1')

try:
    print('Inside outer try: calling func1')
    func1()
finally:
    print('Outer finally')
    # Inside outer try: calling func1
    # Inside func1: calling func2
    # Inside func2
    # Finally in func1
    # Outer finally
    # After that, Python displays the unhandled ValueError traceback.
# 3. Main difference
    # Nested try/except	                                Nested try/finally
    # Handles a matching exception.	                    Executes termination code.
    # Only the most recent matching handler catches it.	Every active finally block executes during propagation.
    # Stops propagation when caught.	                Does not stop propagation.
    # Execution continues after the handling try.	    Exception continues outward unless caught elsewhere
# 4. The key concept: exception propagation
    # The main point is that exception propagation depends on the runtime flow of control, not just the physical arrangement of code.
    # - Python moves backward through previously entered but not yet exited try statements.
    # - Propagation stops when a matching except catches the exception.
    # - Propagation does not stop when Python passes through a finally block.
    # - If no handler catches the exception, it eventually reaches the top-level handler.
# What about except*?
    # The except* clauses do not change this behavior:
    # - If they consume every exception in a group, the exception group ends in that try statement.
    # - Unmatched exceptions are raised again and propagate to other try statements or the top-level handler

# Example: Control-Flow Nesting
    # This example demonstrates how exceptions move through nested function calls and how Python selects the most recently entered matching try statement.
# 1. Nested individual exceptions
    # The program has three functions:
    # - action1() calls action2() inside a try.
    # - action2() calls action3() inside another try.
    # - action3() generates a TypeError
%run nested_exc_normal.py

# 2. Nested exception groups
    # The same nesting principle applies to exception groups, but there is an important difference:
    # each handler can consume only the exceptions it matches. Unmatched exceptions continue propagating.
%run nested_exc_group.py

# 3. Main difference between the two examples
    # Individual exception	                                    Exception group
    # The inner handler catches the whole exception.	        The inner handler consumes only matching exceptions.
    # raise is needed to propagate the caught exception again.	Unmatched exceptions automatically propagate.
    # The outer handler catches the reraised exception.	        The outer handler consumes its matching exceptions.
    # Propagation ends when the exception is caught.	        Propagation ends when all exceptions are consumed.
# 4. Main conclusion
    # Exception routing depends more on runtime control flow than on statement syntax.
    # To understand where an exception goes, you need to know which functions and try statements were entered before the exception occurred


# Example: Syntactic Nesting
    # syntactic nesting: one try statement is physically written inside another try block.
# 1. Nested try/except
from nested_exc_normal import action3

try:
    try:
        action3()
    except TypeError:       # Most-recent matching try
        print('Inner try')
        raise
except TypeError:           # Here, only if nested handler reraises
    print('Outer try')      # Inner try # Outer try
    # How it works:
    # 1. The outer try contains another try.
    # 2. action3() raises TypeError.
    # 3. The inner except TypeError catches it and prints Inner try.
    # 4. raise reraises the exception.
    # 5. The outer except TypeError catches it and prints Outer try.
    # This behaves exactly like the previous runtime-nesting example. The only difference is that the nested handlers are
    # physically inside the same code block rather than inside separate functions.
# 2. Nested try/finally
    # The same principle applies to finally blocks.
try:
    try:
        action3()
    finally:
        print('Inner try')
finally:
    print('Outer try')  # Inner try # Outer try
    # Then Python displays the TypeError traceback.
    # How it works:
    # 1. action3() raises TypeError.
    # 2. The inner finally executes and prints Inner try.
    # 3. The exception continues propagating.
    # 4. The outer finally executes and prints Outer try.
    # 5. The exception remains unhandled and reaches the top-level handler.

    # 3. Combining except and finally
    # This example demonstrates three different functions:
        # - raise1() raises IndexError.
        # - noraise() returns normally without raising an exception.
        # - raise2() raises SyntaxError.
%run except-finally.py

# 4. Main conclusion
    # nested except and finally statements behave the same whether they are nested syntactically or through function calls at runtime.
    # Syntactic nesting	                                        Runtime nesting
    # try statements are physically inside other try blocks.	Functions call other functions containing try statements.
    # Handlers are embedded in the source code.	                Handlers are located in separate functions.
    # Same exception-propagation behavior.	                    Same exception-propagation behavior.

    # Why is syntactic nesting useful?
    # Although Python allows except and finally in the same try statement, syntactic nesting makes their separate roles explicit:
        # - except handles matching exceptions.
        # - finally performs termination-time actions regardless of whether an exception occurs.
    # It can also be useful for implementing alternative exception-handling behaviors

# Exception Idioms
    # This section explains common ways Python exceptions are used in practical programming.

# Breaking Out of Multiple Nested Loops: “go to”
# 1. Breaking Out of Multiple Nested Loops: “go to”
    # Normally, Python's break statement exits only the closest enclosing loop.
    # However, exceptions can be used to exit several nested loops at once.
    # The book compares this behavior to a goto statement in other programming languages:
    # - raise acts like goto.
    # - except acts like the destination.
    # - A custom exception class identifies where execution should jump.
    # Unlike an unrestricted goto, this approach limits the jump to code enclosed in a try block.
%run breaker.py

# Exceptions Aren’t Always Errors
    # All errors are exceptions, but not all exceptions represent errors.
    # Some exceptions are simply signals used to control program execution.
    # 1. EOFError — End of file
        # Python handles end-of-file differently depending on the method:
        # - File object's read() method returns an empty string ('') at the end of a file.
        # - The built-in input() function raises EOFError at the end of input.
        # An empty string returned by input() means an empty line, not end-of-file.
        # Therefore, EOFError can be used as a signal to stop reading input.
# Example:
while True:
    try:
        line = input()  # Read line from stdin
    except EOFError:
        break           # Exit loop at end-of-file
    else:
        ...             # Process next line here
    # How it works:
    # 1. input() reads a line.
    # 2. If input is available, the else block processes it.
    # 3. If end-of-file is reached, input() raises EOFError.
    # 4. The except handler catches it and break exits the loop.

# 2. Other exceptions used as signals
    # Action	            Exception raised
    # Calling sys.exit()	SystemExit
    # Pressing Ctrl+C	    KeyboardInterrupt
    # These exceptions are also used as signals rather than ordinary programming errors.

# 3. Warnings
    # Python also has built-in exceptions that represent warnings rather than errors.
    # Some warnings signal the use of deprecated features (features that will be phased out).
    # The book refers to:
    # - Built-in exceptions documentation.
    # - The warnings module documentation.

# Functions Can Signal Conditions with raise
    # The functions can use exceptions to signal success or failure instead of returning status flags or special values.
    # The book presents two approaches.
# 1. Using an exception to signal success
    # A search function can raise an exception when it finds a match.
class Found(Exception):
    pass

def searcher():
    if ...:             # success
        raise Found()   # Raise exceptions instead of returning flags
    else:
        return

try:
    searcher()
except Found:           # Exception if item was found
    ...                 # success
else:                   # else returned: not found
    ...                 # failure
    # How it works:
    # - If the search succeeds, searcher() raises Found.
    # - except Found catches it and executes the success code.
    # - If the search does not succeed, the function returns normally.
    # - The else block executes and handles the failure case.
    # Here, the exception acts like a success signal.

# 2. Using an exception to signal failure
    # Another approach is to return the found item when successful and raise an exception when unsuccessful.
class Failure(Exception):
    pass

def searcher():
    if ...:             # success
        return founditem
    else:
        raise Failure()

try:
    item = searcher()
except Failure:
    ...                 # not found
else:
    ...                 # use item here

    # How it works:
    # - If successful, the function returns founditem.
    # - The else block uses the returned item.
    # - If unsuccessful, the function raises Failure.
    # - The except block handles the failure.
# 3. Why use exceptions instead of return values?
    # Sometimes every possible object could be a valid return value.
    # In that situation, a function cannot safely use a special return value (called a sentinel value) to indicate failure.
    # Exceptions solve this problem because they signal a condition without requiring a special return value.

# Closing Files and Server Connections
    # Exceptions can be used to ensure that system resources are closed or finalized, regardless of whether an error occurs.
    # Examples of resources include:
    # - Server connections: must be closed to terminate a session.
    # - Output files: need to be closed to flush buffered data to disk.
    # - Input files: should be closed to release file descriptors.
    # CPython may close open files during garbage collection, but this is not always predictable or reliable.
# 1. Using try/finally
    # The most general and explicit way to guarantee cleanup is try/finally.

myfile = open('somefile', 'w')
try:
    ...  # process myfile
finally:
    myfile.close()
    # How it works:
    # - try executes the file-processing code.
    # - finally always executes when leaving the try block.
    # - myfile.close() closes the file, even if an exception occurs.
# 2. Using with (Context Manager)
    # Some objects provide context managers that automatically close or terminate resources
with open('somefile', 'w') as myfile:
    ...  # process myfile
    # Here, the with statement automatically handles resource cleanup when execution leaves the block.
# 3. Main difference
    # try/finally	                     with
    # Explicitly calls myfile.close().	 Automatically handles resource cleanup.
    # Uses a finally block.	             Uses a context manager.
    # General and explicit approach.	 Potentially easier when the object supports context management

# Debugging with Outer try Statements
    # The idea is to put an outer try around the entire program so that any exception not handled inside the program can be caught.
# 1. Example
try:
    ...run program...
except:                 # All uncaught exceptions come here
    import sys
    print('uncaught!', sys.exc_info()[0], sys.exc_info()[1])
    # try: runs the program.
    # except: catches any exception that was not handled elsewhere.
    # sys.exc_info(): provides information about the exception currently being handled.
# 2. What does sys.exc_info() return?
    # It returns a tuple. The first two elements are:
    # Expression	        Meaning
    # sys.exc_info()[0]	    Exception class (type)
    # sys.exc_info()[1]	    Exception instance (the actual exception object)

    # For example, if the program raises ZeroDivisionError, these expressions give:
sys.exc_info()[0]  # ZeroDivisionError
sys.exc_info()[1]  # division by zero

# 3. Why use this?
    # Normally, an uncaught exception causes Python to terminate the program and display a traceback.
    # With an outer try, you can catch that exception and control what happens next. This is useful during:
    # - Development: keep the program active after an error.
    # - Testing: catch an error and continue running additional tests without restarting the program.

# Running In-Process Tests
    # how to test multiple functions or modules within the same Python process, without stopping the test driver when one test fails.
# 1. Example code
import sys
log = open('testlog', 'a')
from testapi import moreTests, runNextTest, testName

def testdriver():
    while moreTests():
        try:
            runNextTest()
        except:
            print('FAILED', testName(), sys.exc_info()[:2], file=log)
        else:
            print('PASSED', testName(), file=log)

testdriver()
# 2. How does it work?
    # The testdriver() function repeatedly runs tests until moreTests() returns False.
    # For each test:
    # 1. while moreTests() checks whether another test is available.
    # 2. runNextTest() executes the next test.
    # 3. except: catches any uncaught exception raised by that test.
    # 4. If an exception occurs, the driver writes FAILED, the test name, and exception information to the testlog file.
    # 5. else: runs only if runNextTest() finishes without raising an exception. It writes PASSED to the log.

# 3. What does sys.exc_info()[:2] mean?
    # sys.exc_info() returns a tuple containing exception information.
    # - [:2] selects the first two elements.
    # - The first is the exception class.
    # - The second is the exception instance.
    # For example, the log might contain:
    #     FAILED test_division (<class 'ZeroDivisionError'>, ZeroDivisionError('division by zero'))
    #     PASSED test_addition
# 4. Why is the try inside the loop?
    # Because the try is inside the loop, a failed test does not stop the entire test driver. The next iteration can still run.
    # If the try were outside the loop, an uncaught exception could interrupt the whole testing sequence.

# 5. Testing external programs
    # The author then distinguishes in-process testing from testing external programs.
    # - In-process tests run imported functions, modules, and classes directly.
    # - External programs can be tested by checking their status codes or outputs using tools such as os.system and os.popen.
    # - These tools generally do not raise Python exceptions just because the external program encounters an error.
# The book later introduces more complete testing frameworks:
    # - doctest – checks expected outputs against actual outputs.
    # - PyUnit – provides a framework for organizing and running tests.

# More on sys.exc_info
    # sys.exc_info() allows an exception handler to access information about the exception currently being handled.
    # It is particularly useful with an empty except clause, because that clause does not bind the exception to a variable.
# 1. What does sys.exc_info() return?
    # It returns a tuple containing three values:
#(type, value, traceback)

    # Element	Meaning
    # type	    The exception class, e.g. ZeroDivisionError
    # value	    The actual exception instance raised
    # traceback	A traceback object representing the call stack when the exception occurred
    # If no exception is currently being handled, all three values are None.
# 2. Example with an empty except
import sys
try:
    1 / 0
except:
    print(sys.exc_info()[0])
    print(sys.exc_info()[1])
    print(sys.exc_info()[2])
    # The first two results are:
    # <class 'ZeroDivisionError'>
    # division by zero

    # The third is the traceback object.
    # Remember:
    # - [0] gives the exception class.
    # - [1] gives the exception instance.
    # - [2] gives the traceback.

# 3. Is sys.exc_info() necessary with as? Usually, no.
    # You can obtain the exception instance directly using as:
try:
    1 / 0
except Exception as instance:
    print(instance.__class__)
    print(type(instance))
    # Both instance.__class__ and type(instance) give the exception class.
    # Here, instance gives you access to the actual exception object.
# 4. Why use Exception instead of an empty except?
    # Compare:
try:
    ...
except:
    ...
    # with:
try:
    ...
except Exception as instance:
    ...
    # The second approach catches ordinary, non-exit exceptions but is less extreme than an empty except.
    # It also gives direct access to the exception instance through instance
# 5. Polymorphism versus checking exception types
    # The one final point: instead of checking which exception class was raised, it is often better to call a method on the exception instance
except Exception as instance:
    instance.method()
    # The method can perform the appropriate behavior for that particular exception object.

# The sys.exception alternative—and diss
    # This section introduces sys.exception(), added in Python 3.11, as another way to obtain information about the currently handled exception.
# 1. What does sys.exception() return?
    # It returns only the exception instance.
    # It is equivalent to:  sys.exc_info()[1]
# For example:
import sys

try:
    1 / 0
except:
    print('uncaught!', type(sys.exception()), sys.exception()) # uncaught! <class 'ZeroDivisionError'> division by zero
    # Notice that:
    # - sys.exception() returns the exception instance.
    # - type(sys.exception()) returns its exception class.
# 2. Three ways to get the same information
    # The book demonstrates three equivalent approaches:
import sys

class E(Exception):
    pass

try:
    raise E('info')
except E as X:
    print((type(X), X))
    print(sys.exc_info()[:2])
    print((type(sys.exception()), sys.exception()))
    # (<class '__main__.E'>, E('info'))
    # (<class '__main__.E'>, E('info'))
    # (<class '__main__.E'>, E('info'))

    # Approach	                                Exception class	        Exception instance
    # type(X), X	                            type(X)	                X
    # sys.exc_info()[:2]	                    First element	        Second element
    # type(sys.exception()), sys.exception()	type(sys.exception())	sys.exception()
    # Here, X is the variable assigned by except E as X.

# 3. What about the traceback?
    # With sys.exception(), you can also access the traceback through the exception instance:
sys.exception().__traceback__
    # So, you can obtain all three pieces of information:
    # - Exception class: type(sys.exception())
    # - Exception instance: sys.exception()
    # - Traceback: sys.exception().__traceback__
# 4. Why does the author criticize sys.exception()?
    # The Python documentation describes sys.exc_info() as an older approach.
    # The author disagrees with treating it as outdated because:
    # - sys.exception() provides almost the same information.
    # - Its main advantage is avoiding [1] when only the exception instance is needed.
    # - sys.exc_info() is already traditional and widely used.
    # The author considers replacing the older call for such a small convenience unnecessary.


# Displaying Errors and Tracebacks
    # how to catch an exception but still save its complete error message and traceback using Python's traceback module.
# 1. Example code
%run badly.py
!cat badly.txt

# Exception Design Tips and Gotchas
    # Common mistakes when using exceptions often come from design decisions, rather than difficulties with Python syntax.
    # There are two important decisions when designing exception handling:
# 1. How specific or general should except clauses be?
    # - Should you catch only a particular exception, such as ValueError?
    # - Or should you catch a broader category of exceptions?
# 2. How much code should be placed inside a try block?
    # - Should the try block contain only a small piece of code?
    # - Or should it wrap a larger section of the program?
    # These decisions are the real art of using exceptions effectively.

# What Should Be Wrapped
    # You do not need to put every statement inside its own try block. Deciding which code to wrap depends on the purpose and design of the program.
    # Rules of thumb
# 1. Wrap operations that commonly fail.
    # Operations that interact with the system are good candidates for try:
    # - Opening files.
    # - Making socket calls.
    # - Other operations involving system state.
# 2. Do not always catch failures.
    # Sometimes a failure should stop the program rather than be caught and ignored.
    # For example, if a failure makes the program unable to continue, allowing it to terminate may be better than hiding the problem.
    # Python normally displays useful error messages when an exception is not caught.
# 3. Use try/finally for cleanup actions.
    # If cleanup must happen regardless of whether an exception occurs, use try/finally, unless a context manager (with) is available.
# 4. Consider wrapping a function call instead of its internal operations.
    # Instead of putting separate try blocks around operations inside a function, you can wrap the function call itself.
# For example:
def process():
    open_file()
    read_file()
    process_data()

try:
    process()
except Exception:
    print('An error occurred')
    # Exceptions raised inside process() can propagate to the single try surrounding the function call.

    # Exception handling depends on the program type
    # Program type	    Typical exception handling
    # Servers	        Catch and recover from exceptions
    # Test runners	    Catch exceptions and continue testing
    # GUIs	            Catch and recover from exceptions
    # Simple scripts	Often allow exceptions to terminate the program
    # Simple scripts often do not need extensive exception handling because a failure at any step may require the entire program to stop

# Catching Too Much: Avoid Empty except and Exception
    # Catching every exception can hide important errors and prevent the program from behaving correctly.
# 1. Empty except can catch exceptions meant for another handler
# Consider:
def func():
    try:
        # IndexError occurs here
        ...
    except:
        # Catches everything
        ...

try:
    func()
except IndexError:
    # Expected to handle IndexError here
    ...
    # The problem:
    # 1. func() raises IndexError.
    # 2. Its internal empty except catches the exception.
    # 3. The outer except IndexError never receives it.
    # The exception is caught too early
# 2. Empty except can prevent program termination
    # Python's sys.exit() raises a SystemExit exception to terminate the program.
# Example:
%run exiter.py

# 3. Does except Exception solve this problem?
    # It avoids catching SystemExit, unlike an empty except.
try:
    bye()
except Exception:
    ...
    # However, Exception still catches many other exceptions, including programming errors.
    # Therefore, it can still hide mistakes in your code.

# 4. Example: A misspelled variable
# Consider:
mydictionary = {'name': 'John'}
try:
    x = myditctionary['age']  # Misspelled variable name
except:
    x = None
print(x) # None
    # What actually happens?
    # 1. The dictionary variable is named mydictionary.
    # 2. The code mistakenly uses myditctionary.
    # 3. Python raises NameError because that name is undefined.
    # 4. The empty except catches the NameError.
    # 5. x = None executes.
    # The programmer expected a missing dictionary key (KeyError), but the actual problem was a misspelled variable (NameError).
    # Even except Exception would produce the same result here because NameError is a subclass of Exception.

# 5. Better approach: Catch specific exceptions
# Instead of:
try:
    x = mydictionary[key]
except:
    x = None
# Use:
try:
    x = mydictionary[key]
except KeyError:
    x = None
    # Now:
    # - If the key does not exist, KeyError is caught and x becomes None.
    # - If the variable name is misspelled, NameError is not caught.
    # - Python displays the programming error instead of silently hiding it.

    # Exception handling	    What it catches	                    Main risk
    # except:	                All exceptions	                    Can catch SystemExit and programming errors
    # except Exception:	        Most ordinary exceptions	        Can still hide programming errors
    # except KeyError:	        Only KeyError and its subclasses	Much less likely to hide unrelated errors

# Catching Too Little: Use Class-Based Categories
    # Being too specific in exception handlers can also cause problems.
try:
    ...
except (MyExcept1, MyExcept2):
    ...
    # This handler catches only MyExcept1 and MyExcept2.
    # If the system later introduces MyExcept3, this handler will not catch it. You would need to update the exception list in your code.
# 2. Catching a common superclass
    # Instead, you can catch a general superclass shared by these exceptions:
try:
    ...
except CommonCategoryName:
    ...
    # If MyExcept3 is later added as a subclass of CommonCategoryName, the existing handler will catch it automatically.
    # Main takeaway
    # - Too general: you may catch unrelated exceptions.
    # - Too specific: you may miss new exceptions added later.
    # - Better design: use an appropriate common superclass to group related exceptions.

# Core Language Wrap-Up
    # Core language: Built-in types, statements, and exceptions.
    # Program organization: Functions, modules, and classes.
    # Software design: Important design principles and program architecture.
    # Object-oriented programming (OOP): The complete OOP paradigm.
    # Functional programming: Tools and techniques for functional programming.
    # Alternative tools: Different programming approaches and their trade-offs.

# The Python Toolset
    # Python has a large and continuously growing collection of tools.
    # For example, its standard library contains hundreds of modules, and many additional tools are available publicly.
# 1. Python's hierarchy of toolsets
    # The text identifies three levels:
    # 1. Built-in tools
        # These are already available in Python, such as:
        # - Strings
        # - Lists
        # - Dictionaries
        # They allow you to create simple programs quickly.
    # 2. Python-coded extensions
        # When programs become more demanding, you can create your own:
        # - Functions
        # - Modules
        # - Classes
        # These are written using Python itself.
    # 3. Other-language extensions
        # Python can also be extended using code written in other programming languages, such as:
        # - C
        # - C++
        # - Java
# 2. How do these levels work together?
        # You can choose how deeply to use this hierarchy depending on the task.
        # Task	                Toolset
        # Simple scripts	    Built-in tools
        # Larger applications	Python-coded functions, modules, and classes
        # Advanced tasks	    Extensions written in other languages
# 3. What does this mean for your Python learning?
    # The book has covered the first two categories:
        # 1. Built-in Python tools.
        # 2. Creating your own functions, modules, and classes.

# Development Tools for Larger Projects
    # Larger Python projects need additional tools beyond basic modules and classes.
# 1. Documentation tools
    # - PyDoc: Generates documentation for Python modules and objects using docstrings.
# 2. Error-checking tools
    # - PyChecker, Pylint, Pyflakes: Detect common programming errors before running the program.
# 3. Testing tools
    # - unittest (PyUnit): Organizes test cases and expected results using classes.
    # - doctest: Extracts examples from docstrings and runs them to check expected results.
# 4. IDEs
    # - PyCharm, IDLE: Provide graphical environments for editing, running, debugging, and browsing code.
    # - Advanced IDEs also support source control, code refactoring, and project management.
# 5. Profilers
    # - profile, cProfile: Identify performance bottlenecks and provide performance statistics.
    # - time, timeit: Measure execution time.
    # Recommended approach: write clear code first, profile it to find slow parts, then compare alternative implementations.
# 6. Debuggers
    # - pdb: Python's command-line debugger.
    # - IDEs such as IDLE provide graphical debugging interfaces.
    # - Programmers also commonly use error messages and print() statements to find problems.
# 7. Shipping options
    # Tools package Python programs into standalone executables that do not require Python to be installed on the target machine.
    #
    # Python programs can also be distributed as:
    # - .py source files
    # - .pyc bytecode files
    # - .zip packages
    # pip is used to install Python packages.
# 8. Optimization options
    # Tools and approaches for improving performance:
    # - PyPy: Provides automatic speed improvements.
    # - Shed Skin and Cython: Offer alternative approaches to faster programs.
    # - C extensions: Move parts of a program to C for better performance.
    # - Python upgrades may also improve speed.
    # The -O flag provides only modest optimization.
# 9. Installation management
    # - venv: Creates independent virtual environments.
    # - Each environment has its own installed Python packages.
    # - pip installs packages into the currently activated environment.
# 10. Other useful tools for larger projects
    # The book also highlights previously covered Python features:
    # - Module packages
    # - Exception classes
    # - Pseudoprivate class attributes
    # - Documentation strings (docstrings)
    # - Module data hiding
    # - Design guidelines for functions, modules, classes, objects, and exceptions